In [32]:
import numpy as np

# LSTM - Theory
---
**Regular Neural Networks Have No Memory**
Imagine you're reading a sentence: "The clouds are in the ___"

A normal neural network sees each word independently - it doesn't remember that you just read "clouds" and "sky" would make sense there.

### RNNs: Networks with Memory
RNNs have a loop that passes information from previous steps to the current step:

text
      hₜ₋₁ ──→ hₜ ──→ hₜ₊₁
       ↑      ↑      ↑
      xₜ₋₁   xₜ    xₜ₊₁
At each step, the RNN:

Takes current input (xₜ)
Takes previous hidden state (hₜ₋₁) - this is the "memory"
Combines them to produce new output and new memory

### The RNN Problem: Short Memory
Simple RNNs forget quickly. It's like having a 3-second memory - you can't remember the beginning of a long sentence.

Example: "I grew up in France... I speak fluent ___" - You need to remember "France" from 10 words ago!

## Enter LSTM: Long Short-Term Memory
LSTM has two memory channels instead of one:

**Hidden state (h)** - short-term memory (forgets quickly)

**Cell state (c)** - long-term memory (holds info longer)

Think of it like a notebook:

Cell state = permanent notes you keep

Hidden state = what you're currently thinking about

**The 3 Gates (Decision Makers)**
LSTM has 3 "gates" that decide what to do with information. Each gate is just a sigmoid function that outputs 0-1 (0 = let nothing through, 1 = let everything through):

1. Forget Gate - "What should I erase?"
text
fₜ = sigmoid(Wf · [hₜ₋₁, xₜ] + bf)
Looks at previous hidden state + current input

Decides what to delete from long-term memory (cell state)

2. Input Gate - "What new info should I save?"
text
iₜ = sigmoid(Wi · [hₜ₋₁, xₜ] + bi)
c̃ₜ = tanh(Wc · [hₜ₋₁, xₜ] + bc)  # candidate values
First decides WHICH values to update (iₜ)

Then creates WHAT new values to add (c̃ₜ)

3. Output Gate - "What should I output?"
text
oₜ = sigmoid(Wo · [hₜ₋₁, xₜ] + bo)
hₜ = oₜ * tanh(cₜ)
Decides what to output based on current cell state

This becomes the hidden state (short-term memory)

The Flow of Information
text
        ←───────── Long-term memory (c) ─────────→
        
Step 1:   c₀ ─────────────────────────→ c₁
           ↑                            ↑
    [Forget] [Input]            [Update cell]
    
          h₀ ──→ [Output] ──→ h₁
           ↑         |
          x₁         └──→ final output

        ←── Short-term memory (h) ──→

Your Code's Role
Your __init__ function sets up:

Weights (W): How strongly each input affects decisions

Biases (b): Baseline preferences for each gate

Each gate has its own weights because they learn different patterns:

A. **Forget gate** learns: "When to erase"

B. **Input gate** learns: "When to save"

C. **Output gate** learns: "When to output"

In [33]:
# Helper Functions

# 1. Sigmoid Function - 1/(1+e^(-x))
def sigmoid(x):
    return (1/(1+np.exp(-x)))

# 2. derivative of Sigmoid Function - y * (1 - y) - where y = sigmoid(x)
def dsigmoid(y):
    return y * (1 - y)

# 3. tanh function - (e^x - e^(-x))/(e^x + e^(-x))
def tanh(x):
    return np.tanh(x)

# 4. derivative of tanh Function - (1 - y**2) - where y = tanh(x)
def dtanh(y):
    return 1 - y**2

In [34]:
class LSTM:
    def __init__(self, input_size, hidden_size, output_size):
        self.hidden_size = hidden_size
        self.input_size = input_size

        # Xavier initialization
        def __init__gate():
            return np.random.randn(hidden_size, hidden_size + input_size) * 0.1

        # Gates
        self.Wf = __init__gate()   # forget gate: remove infor
        self.Wi = __init__gate()   # input gate: decide what to add
        self.Wc = __init__gate()   # Candidate: new info
        self.Wo = __init__gate()   # Output gate: what to output

        # Biases of each gate - biases tends the neuron to either to strongly activation (positive bias) or very weak activation (negative bias)4
        # at first, they are just a 1d array of hidden_size contains all zeros
        self.bf = np.zeros((hidden_size, 1))
        self.bi = np.zeros((hidden_size, 1))
        self.bc = np.zeros((hidden_size, 1))
        self.bo = np.zeros((hidden_size, 1))

        # ---------- Output Layer ---------- #
        self.Wy = np.random.randn(output_size, hidden_size) * 0.1
        self.by = np.zeros((output_size, 1))

# For example, if we consider a output of one gate(forget gate), ft = sigmoid(Wf[ht−1,xt] + bf)
# output = weighted sums of input + bias

# here, inputs are h_{t-1} and x_t, and we taking weighted sums (here, Wf) of inputs plus bias (here, bf) which we passing through sigmoid function.

# we just initiated the class LSTM

## Full Flow Visualization for forward function

```
At first, we need a cache list which is empty at first, which needs for backprop later on
      ↓
x_t + h_{t-1}
      ↓
   [concat]
      ↓
 ┌───────────────┐
 │ 4 Gates       │
 │               │
 │ f → forget    │ -> f = sigmoid(Wf.[h_t-1, x_t] + bf)
 │ i → write     │ -> i = sigmoid(Wi.[h_t-1, x_t] + bi)
 │ c~ → content  │ -> c~ = tanh(Wc.[h_t-1, x_t] + bc)
 │ o → output    │ -> o = sigmoid(Wo.[h_t-1, x_t] + bo)
 └───────────────┘
      ↓   (previously, there's the memory present for long time)
 Update memory:
 c = f*c + i*c~   --> c_t = f_t * c_t-1 + i_t * c~_t
      ↓
 Output:
 h = o * tanh(c)
      ↓
 cache appended with (h, c, f, i, c_tilde, o, [h_t-1, x_t]) 
 that means all gate values, all memory values and inputs
      ↓
return h and c

In [35]:
# To feed inputs [h_t-1, x_t] to the system, we need to make it one matrix, that means we will vertically stack two inputs to make one input, that input we will be calling concat

def forward(self, inputs):
    h = np.zeros((self.hidden_size, 1))
    c = np.zeros((self.hidden_size, 1))

    self.cache = []

    for x in inputs:
        x = x.reshape(-1,1)   # To vertically stack elements one above another, we need to make it column matrix first
        concat = np.vstack((h, x))  # --> [h_t-1, x_t]

        # calculating all gates
        f = sigmoid(self.Wf @ concat + self.bf)
        i = sigmoid(self.Wi @ concat + self.bi)
        c_tilde = tanh(self.Wc @ concat + self.bc) 
        o = sigmoid(self.Wo @ concat + self.bo)

        # updating memory
        c = f * c + i * c_tilde

        # final memory output h 
        h = o * tanh(c)

        # appending all the details in cache -> will help in backdrop
        self.cache.append((h, c, f, i, c_tilde, o, concat))

    # ---------- Final Prediction ---------- #
    y = self.Wy @ h + self.by
    
    return y, h, c       

LSTM.forward = forward
# Questions Arised:- 
    # 1. Why do we take tanh for c_tilde instead of sigmoid function?
    # Ans:- we taking tanh function instead of sigmoid, to take the negative values too, cuzz memories have both negative and positive context

    # 2. What is the use of cache? 
    # Ans:- cache storing screenshot of what forward function recorded for each time (or each inputs), these are necessary for backpropagation.

## Backward Function

Forward pass: input → output  
But learning needs: “How wrong was I?” → “How should I change weights?”  

That’s **backward pass**.

Backward pass computes gradients (how each weight should change)  
Without it:  
❌ weights never update  
❌ model never improves  

🧠 Simple analogy  
**Forward pass**:  You take a test  
**Backward pass**: You check mistakes and learn  

This function takes two Inputs:
- **dh_next** → gradient coming from future (loss wrt h)
- **dc_next** → gradient wrt memory

In Short, Go BACKWARD through time:
t = last → first

At each step:
- compute gradients of gates
- compute gradients of weights
- pass gradients to previous timestep

This is called **Backpropagation Through Time (BPTT)**

### Initialize gradients 
```
dWf = np.zeros_like(self.Wf)
...
dbf = np.zeros_like(self.bf)
```

We accumulate gradients of all weights and biases at each timesteps  

### Loop Backward in Time
```
for t in reversed(range(len(self.cache))):
```
If forward was: t1 → t2 → t3  
Backward is: t3 → t2 → t1

### Retrieve stored values
```
h, c, f, i, c_tilde, o, concat = self.cache[t]
```
- This is why cache exists
- You need these values to compute gradients

**First:** The ONLY rule you need

Everything here follows one idea:

**Chain Rule**:
If ```y=f(g(x))```, then

$\frac{dx}{dy} = \frac{dg}{dy}.\frac{dx}{dg}$

🎯 Start from what we know (forward equations)

At one timestep:

1. ```h = o⋅tanh(c)```
2. ```c_t = f_t * c_t-1 + i_t * c~_t```
3. ```f=σ(...)   i=σ(...)   o=σ(...)    c~=tanh(...)```

🔁 We go BACKWARD

We are given:
```
dh_next = dL/dh
dc_next = dL/dc (from future)
```
### Step A: Output gate gradient (do)

Forward: ```h=o⋅tanh(c)``` $\implies \frac{dh}{do} = tanh(c)$

Step-by-step:

We want: $\frac{dL}{do}$ 

Using chain rule: $\frac{dL}{do}= \frac{dL}{dh}.\frac{dh}{do}$

So: ```do = dh_next * tanh(c)```

BUT o came from sigmoid activation, we need raw gate value before sigmoid (dL/dz)  
```o = sigmoid(z)``` $\implies \frac{do}{dz} = o *(1 - o) $

Using chain rule: $\frac{dL}{dz}= \frac{dL}{do}.\frac{do}{dz} \implies \frac{dL}{dz} = do * dsigmoid(o)$
$\implies \frac{dL}{dz} = dh_{next} * tanh(c) * dsigmoid(o)$

replacing $\frac{dL}{dz}$ by do 

So final: ```do = dh_next * tanh(c) * dsigmoid(o)```

### Step B: Gradient of Cell State (`dc`)

Path 1 (through hidden state):
$
h = o \cdot \tanh(c)
\implies
\frac{dh}{dc} = o \cdot sech^2(c)
\implies
\frac{dh}{dc} = o \cdot (1 - \tanh^2(c))
$

Path 2 (from future timestep):
$
dc_{\text{next}}
$

Combine:
$
\frac{dL}{dc} = \frac{dL}{dh}.\frac{dh}{dc} + dc_{\text{next}} \implies dc = dh_{\text{next}} \cdot o \cdot (1 - \tanh^2(c)) + dc_{\text{next}}
$  
Two gradients flow merge here

### Step C: Gradient of Candidate Memory (`dc_tilde`)

$
c_{t} = f \cdot c_{t-1} + i \cdot \tilde{c}  
$  
Applying derivative in the first part with respect to $\tilde{c}$, it becomes zero, so we can neglect that part  
$
c = i \cdot \tilde{c}
\implies
\frac{dL}{d\tilde{c}} = \frac{dL}{dc}.\frac{dc}{d\tilde{c}}
\implies
\frac{dL}{d\tilde{c}} = dc \cdot i
$  
Applying tanh derivative  
$
\implies
dc_{\tilde{}} = dc \cdot i \cdot dtanh(\tilde{c})
\implies
dc_{\tilde{}} = dc \cdot i \cdot (1 - \tilde{c}^2)
$

### Step D: Gradient of Input Gate (`di`)

$
c = i \cdot \tilde{c}
\implies
\frac{dL}{di} = \frac{dL}{dc}.\frac{dc}{di}
\implies
\frac{dL}{di} = dc \cdot \tilde{c}
$  
Applying sigmoid derivative  
$
\implies
dc_{\tilde{}} = dc \cdot i \cdot dsigmoid(i)
\implies
dc_{\tilde{}} = dc \cdot i \cdot i \cdot (1 - i)
$

### Step E: Gradient of Forget Gate (`df`)

$
c_{t} = f \cdot c_{t-1} + i \cdot \tilde{c}  
$  
Applying derivative in the second part with respect to $f$, it becomes zero, so we can neglect that part  
$
c = f \cdot c_{\text{prev}}
\implies
\frac{dL}{df} = \frac{dL}{dc}.\frac{dc}{df}
\implies
\frac{dL}{df} = dc \cdot c_{\text{prev}}
$  
Applying sigmoid derivative  
$
\implies
df = dc \cdot c_{\text{prev}} \cdot dsigmoid(f)
\implies
df = dc \cdot c_{\text{prev}} \cdot f(1 - f)
$

### Step G: Gradient of weights (`W`)

In each gates, we have weigths and biases, basically in form of:- **gate = weight * concat + bias**  
-- here, gate could be forgot gate, input gate, output gate, etc...  

Now, when we going to find the gradient of weights, that means derivative of Loss wrt weights ($\frac{dL}{dW}$)  
$\frac{dL}{dW} = \frac{dL}{dgate}.\frac{dgate}{dW}$  
$\frac{dL}{dgate}$ --> basically, **gradient of gate** (gradient of forward, input, output, etc...) those we calculated above...  

As bias is not dependent on weights, so while deriving gates with weights, that bias term will be zero, so we can neglect that...  
$\frac{dgate}{dW} = concat^T$ --> $concat^T$ means transpose of concat matrix  

So, $dWeight = dgate * concat^T$

### Step H: Gradient of biases (`b`)

In each gates, we have weigths and biases, basically in form of:- **gate = weight * concat + bias**  
-- here, gate could be forgot gate, input gate, output gate, etc...  

Now, when we going to find the gradient of biases, that means derivative of Loss wrt biases ($\frac{dL}{db}$)  
$\frac{dL}{db} = \frac{dL}{dgate}.\frac{dgate}{db}$  
$\frac{dL}{dgate}$ --> basically, **gradient of gate** (gradient of forward, input, output, etc...) those we calculated above...  

As weights is not dependent on biases, so while deriving gates with biases, that weights term will be zero, so we can neglect that...  
$\frac{dgate}{db} = 1$ 

So, $dbiases = dgate * 1 = dgate$

### Step I: Gradient of concantenated matrix of h and x at first (`dconcat`)

each gates associated with concat matrix, basically in form of:- **gate = weight * concat + bias**  
-- here, gate could be forgot gate, input gate, output gate, etc...  

Now, when we going to find the gradient of concat matrix, that means derivative of loss wrt concat ($\frac{dL}{dconcat})
$\frac{dL}{dconcat} = \frac{dL}{dgate}.\frac{dgate}{dconcat}$  
$\frac{dL}{dgate}$ --> basically, **gradient of gate** (gradient of forward, input, output, etc...) those we calculated above...  

As bias is not dependent on concats, so while deriving gates with concats, that bias term will be zero, so we can neglect that...  
$\frac{dgate}{dconcat} = weight^T$ 

So, $dconcat = dgate * weight^T$  

here's one thing, concat matrix comes from 4 gates, so it's gradient will be influence by 4 gates...  
so, $dconcat = (Wf^T * df) + (Wi^T * di) + (Wc^T * dc) + (Wo^T * do)$

### Calculating dh_next and dc_next

remember, concat is a virtual stack of hidden inputs (h) of hidden_size and inputs (x) of input_size...  

h = $
\begin{bmatrix}
h_1 \\
h_2 \\
h_3
\end{bmatrix}
$
and x = $
\begin{bmatrix}
x_1 \\
x_2 
\end{bmatrix}
$
then concat = $
\begin{bmatrix}
h_1 \\
h_2 \\
h_3 \\
x_1 \\
x_2 
\end{bmatrix}
$
that's why dconcat = $
\begin{bmatrix}
dh_1 \\
dh_2 \\
dh_3 \\
dx_1 \\
dx_2 
\end{bmatrix}
$  

So to find, $dh_{next} = dconcat[:hidden_size, :]$, we just need the dh ones, that means we just need first hidden_size (here, which is 3)

and also we know that $c_t = f \cdot c_{t-1}$, then $dc_{next} = f \cdot c$

### **Final Steps**:-----> Updating weights and biases
-- We know the absolute parameters of weights and biases  
-- We calculated the change or gradient parameters of weight and biases

for updating weights,  
$W_{new} = W - lr \cdot dW$    here, lr means learning rate, which we taking as attribute of the function  

for updating biases,  
$b_{new} = b - lr \cdot db$    here, lr means learning rate, which we taking as attribute of the function  

In [36]:
    def backward(self, dy, dh_next, dc_next, lr=0.01):
        dWf = np.zeros_like(self.Wf)
        dWi = np.zeros_like(self.Wi)
        dWc = np.zeros_like(self.Wc)
        dWo = np.zeros_like(self.Wo)

        dbf = np.zeros_like(self.bf)
        dbi = np.zeros_like(self.bi)
        dbc = np.zeros_like(self.bc)
        dbo = np.zeros_like(self.bo)


        # ---------- Output Layer Gradient ---------- #
        
        dWy = np.zeros_like(self.Wy)
        dby = np.zeros_like(self.by)

        h_last = self.cache[-1][0]

        dWy += dy @ h_last.T
        dby += dy

        dh_next += self.Wy.T @ dy
        
        # ------------------------------------------- #

        for t in reversed(range(len(self.cache))):
            h, c, f, i, c_tilde, o, concat = self.cache[t]

            tanh_c = tanh(c)

            # output gate derivative
            do = dh_next * tanh_c
            do = do * dsigmoid(o)

            # cell gate derivative
            dc = dh_next * o * dtanh(tanh_c) + dc_next

            # c~ gate derivative
            dc_tilde = dc * i
            dc_tilde = dc_tilde * dtanh(c_tilde)

            # input gate derivative
            di = dc * c_tilde
            di = di * dsigmoid(i)

            # forget gate derivative
            df = dc * c
            df = df * dsigmoid(f)

            dWf += df @ concat.T
            dWi += di @ concat.T
            dWc += dc_tilde @ concat.T
            dWo += do @ concat.T

            dbf += df
            dbi += di
            dbc += dc_tilde
            dbo += do

            dconcat = (
                self.Wf.T @ df +
                self.Wi.T @ di +
                self.Wc.T @ dc_tilde +
                self.Wo.T @ do
            )

            dh_next = dconcat[:self.hidden_size, :]
            dc_next = f * dc

        # Update weights
        for param, dparam in zip(
            [self.Wf, self.Wi, self.Wc, self.Wo,
             self.bf, self.bi, self.bc, self.bo, self.Wy, self.by],
            [dWf, dWi, dWc, dWo,
             dbf, dbi, dbc, dbo, dWy, dby]
        ):
            param -= lr * dparam

LSTM.backward = backward

## Training

### Initialisation
-- At first initiating the LSTM class with input_size and hidden_size  
   here, input_size = 1, that means, input feature is only 1  
   and, hidden_size = 16, means memory vector will consists 16 feature embeddings  

-- What the loop does?  
   1. Give sequence
   2. Predict next value
   3. Measure error
   4. Backpropagate error
   5. Update weights
   6. Repeat

### training loop

here, will train it 100 times on same sequence, that means  
```forward --> backward --> update (x100)```


### input sequence 
creating input sequence where each input will be only 1 value, as size is 1  
here, we took $1 -> 2 -> 3 -> 4$  

### forward pass
```h, c = lstm.forward(inputs)```  

**What happens Internally?**  
x1=1 → update   
x2=2 → update memory  
x3=3 → update memory  
x4=4 → final hidden state  

h represents the model understanding of the sequence at the final step, so it will have knowledge of next number in sequence  

### Target output
for example, we set a target output manually that it will be '5' after 4

### Compute loss
```loss = (h - target)^2```  

Meaning -- Measure prediction error.  

**point to be noted**  
-- Your h shape is: (16,1) because hidden size = 16.  
-- But target shape is: (1,1)  
-- NumPy broadcasts automatically.  
-- So actually: all 16 hidden neurons are being compared to 5  

### compute loss gradient

```dh = 2 * (h - target)```

Meaning:-
This tells: ```“How should hidden state change to reduce error?”```

### initial cell state gradient

```dc = np.zeros_like(c)```  

Why zero? At final timestep: no future cell state exists  
So: ```initial future memory gradient = 0```

### Backward pass
```lstm.backward(dh, dc, lr=0.001)```

Backward pass:  
-- computes gradients  
-- propagates through time  
-- updates weights  

### Print progress

print losses to see whether it learning or not...

In [37]:
# Example sequence: learn identity
lstm = LSTM(input_size=1, hidden_size=16, output_size = 1)

for epoch in range(500):
    inputs = [np.array([i]) for i in [1, 2, 3, 4]]

    y, h, c = lstm.forward(inputs)

    target = np.array([[5]])  # expected next
    loss = (y - target)**2

    dy = 2 * (y - target)
    dh = np.zeros_like(h)
    dc = np.zeros_like(c)

    lstm.backward(dy, dh, dc, lr=0.001)

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.mean()}")

Epoch 0, Loss: 25.1855736769001
Epoch 10, Loss: 23.752177413339133
Epoch 20, Loss: 22.386644127161457
Epoch 30, Loss: 21.062670542500456
Epoch 40, Loss: 19.76047764073284
Epoch 50, Loss: 18.466228725810836
Epoch 60, Loss: 17.171587168665216
Epoch 70, Loss: 15.87326598871045
Epoch 80, Loss: 14.572471963222174
Epoch 90, Loss: 13.27426689623904
Epoch 100, Loss: 11.986934216454914
Epoch 110, Loss: 10.721379931501154
Epoch 120, Loss: 9.490492683497473
Epoch 130, Loss: 8.308347987759808
Epoch 140, Loss: 7.189197738436086
Epoch 150, Loss: 6.146297183712577
Epoch 160, Loss: 5.190725856002709
Epoch 170, Loss: 4.330407222765435
Epoch 180, Loss: 3.5695049454382506
Epoch 190, Loss: 2.9082867528312395
Epoch 200, Loss: 2.3434383671101653
Epoch 210, Loss: 1.8687200753901436
Epoch 220, Loss: 1.4758120341839223
Epoch 230, Loss: 1.1551942738333134
Epoch 240, Loss: 0.8969406528544941
Epoch 250, Loss: 0.6913538565202568
Epoch 260, Loss: 0.5294147215780308
Epoch 270, Loss: 0.4030541258945937
Epoch 280, Los